# BƯỚC 3 — PREPROCESSING: Làm sạch & tích hợp dữ liệu
**Đề tài 5:** Dự đoán tái nhập viện trong 30 ngày · **Không dùng cloud**.

**Câu hỏi nghiệp vụ:** Sau khi xuất viện, nên gọi theo dõi cho bệnh nhân nào trước?

**Nguyên tắc:** Giữ nguyên file gốc; không xóa tùy tiện lượt khám, không biến trạng thái không xét nghiệm thành kết quả bình thường, không fit bộ điền khuyết/chuẩn hóa trên toàn bộ dữ liệu. Những biến được xử lý có nhật ký quyết định. Đầu ra *cleaned* phục vụ bước tạo đặc trưng/kho dữ liệu, **chưa phải** X_train/X_test.

## 1. Đọc dữ liệu từ Bước 1, kiểm tra tính toàn vẹn

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

cwd = Path.cwd()
root = next((p for p in [cwd, *cwd.parents] if (p / 'data' / 'processed' / 'step1_data.csv').exists()), None)
if root is None:
    raise FileNotFoundError('Không thấy data/processed/step1_data.csv. Mở JupyterLab từ D:/Readmission_Project.')
src = root / 'data' / 'processed' / 'step1_data.csv'
out_data = root / 'data' / 'processed'
out_report = root / 'reports' / 'preprocessing'
out_report.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(src, low_memory=False)
expected_cols = {'encounter_id','patient_nbr','readmitted','readmitted_30d','weight','payer_code','medical_specialty','A1Cresult','max_glu_serum'}
assert expected_cols.issubset(df.columns), f'Thiếu cột: {expected_cols-set(df.columns)}'
print('Nguồn:', src)
print('Kích thước:', df.shape)
print('Bệnh nhân duy nhất:', df.patient_nbr.nunique())
print('Encounter ID trùng:', df.encounter_id.duplicated().sum())
display(df.head(3))

## 2. Kiểm tra nhãn và đối chiếu với báo cáo EDA

Nhóm `>30` có tái nhập viện sau 30 ngày nhưng vẫn mang nhãn 0 theo **định nghĩa nhị phân**. Không gọi toàn bộ nhãn 0 là "không tái nhập viện".

Báo cáo EDA là đầu vào tham khảo; bảng kiểm tra được tính lại từ dữ liệu thực tế.

In [ ]:
assert df['readmitted'].isin(['<30','>30','NO']).all(), 'Có nhãn readmitted không hợp lệ: cần kiểm tra trước khi tiếp tục.'
computed_target = df['readmitted'].eq('<30').astype('int8')
mismatch = int((pd.to_numeric(df['readmitted_30d'], errors='coerce') != computed_target).sum())
assert mismatch == 0, f'Nhãn sai tại {mismatch} dòng. Dừng để xem lại Bước 1.'
print('Kiểm tra mapping nhãn: 0 dòng sai')
print(df['readmitted'].value_counts(dropna=False).to_string())
print('\nTỷ lệ dưới 30 ngày:', round(100*computed_target.mean(), 2), '%')

eda_report = root / 'reports' / 'eda' / 'data_quality_findings.csv'
if eda_report.exists():
    print('\nĐã tìm thấy báo cáo EDA:', eda_report)
    display(pd.read_csv(eda_report).head(10))
else:
    print('Chưa thấy reports/eda/data_quality_findings.csv (không bắt buộc để chạy Bước 3).')

## 3. Chuẩn hóa ký hiệu thiếu, giữ nghĩa của trạng thái xét nghiệm

In [ ]:
clean = df.copy(deep=True)
# Chỉ chuyển dấu ? và chuỗi rỗng thành missing; tránh biến các danh mục hợp lệ thành NaN.
string_cols = clean.select_dtypes(include=['object','string']).columns.tolist()
audit_missing = []
for col in string_cols:
    vals = clean[col].astype('string').str.strip()
    q = int(vals.eq('?').fillna(False).sum())
    blank = int(vals.eq('').fillna(False).sum())
    audit_missing.append({'cot':col, 'so_dau_hoi':q, 'so_chuoi_rong':blank})
    clean[col] = vals.mask(vals.isin(['?', '']), pd.NA)
missing_symbols = pd.DataFrame(audit_missing).sort_values('so_dau_hoi', ascending=False)
display(missing_symbols[missing_symbols[['so_dau_hoi','so_chuoi_rong']].sum(axis=1)>0])
missing_symbols.to_csv(out_report / 'missing_symbols_before_cleaning.csv', index=False, encoding='utf-8-sig')

# "None" là nhãn gốc cho không ghi nhận xét nghiệm, KHÔNG phải xét nghiệm bình thường.
for col in ['A1Cresult','max_glu_serum']:
    clean[col] = clean[col].fillna('None')
    clean[col + '_tested'] = clean[col].ne('None').astype('int8')
    print(col, '— phân bố sau chuẩn hóa:')
    print(clean[col].value_counts(dropna=False).to_string())

## 4. Các cột thiếu nhiều: chọn chính sách tường minh

- `weight`: loại **khỏi bản phân tích** do thiếu gần 97%; file đầu vào không bị thay đổi.
- `medical_specialty`, `payer_code`: dùng `Unknown` để bảo toàn "không rõ"; không suy diễn chuyên khoa hay người chi trả.
- `race`: giữ `Unknown` để phục vụ kiểm tra công bằng; không tự dự đoán chủng tộc.
- Chẩn đoán `diag_1`, `diag_2`, `diag_3`: giữ `Unknown` riêng nếu không rõ mã.
- `A1Cresult`/`max_glu_serum`: giữ `None` và thêm cờ `*_tested`.

Các chính sách này là quyết định ban đầu cần báo cáo; **không phải** khẳng định chúng tối ưu cho mọi mô hình.

In [ ]:
before_weight_na = int(clean['weight'].isna().sum())
clean = clean.drop(columns=['weight'])
fill_unknown_cols = ['medical_specialty','payer_code','race','diag_1','diag_2','diag_3']
fill_counts = {}
for col in fill_unknown_cols:
    fill_counts[col] = int(clean[col].isna().sum())
    clean[col] = clean[col].fillna('Unknown')
print('Loại weight khỏi bản đã làm sạch; số giá trị thiếu trước khi loại:', before_weight_na)
print('Số ô đã gắn nhãn Unknown:', fill_counts)
print('Số cột hiện tại:', clean.shape[1])

## 5. Kiểm tra giá trị bất thường — chỉ gắn cờ, không xóa

Không coi ngoại lệ IQR là dữ liệu sai. Một số giá trị lớn có thể đại diện bệnh nhân nhiều lần nhập viện — đối tượng cần quan tâm nhất.

In [ ]:
# Giới hạn miền hợp lý theo cấu trúc bộ dữ liệu, cần xem lại nếu nguồn dữ liệu thay đổi.
range_rules = {'time_in_hospital':(1,14), 'num_lab_procedures':(0,None),
 'num_procedures':(0,None),'num_medications':(0,None),'number_diagnoses':(0,None),
 'number_outpatient':(0,None),'number_emergency':(0,None),'number_inpatient':(0,None)}
issues=[]
for col,(lo,hi) in range_rules.items():
    s = pd.to_numeric(clean[col], errors='coerce')
    invalid = s.isna() | (s < lo)
    if hi is not None:
        invalid |= (s > hi)
    issues.append({'cot':col,'gioi_han_thap':lo,'gioi_han_cao':hi, 'so_gia_tri_bat_hop_le':int(invalid.sum()),
                   'gia_tri_min':s.min(),'gia_tri_max':s.max()})
range_report=pd.DataFrame(issues)
display(range_report)
range_report.to_csv(out_report/'range_checks.csv', index=False,encoding='utf-8-sig')

categorical_checks = {col: clean[col].value_counts(dropna=False).to_dict()
                      for col in ['gender', 'age', 'readmitted']}
for col, counts in categorical_checks.items():
    print(f'\n{col}:', counts)

# Những lượt khám cùng patient_nbr KHÔNG bị xem là trùng bản ghi.
print('Duplicate encounter_id:', int(clean['encounter_id'].duplicated().sum()))
print('Duplicate full rows:', int(clean.duplicated().sum()))
print('Lượt vượt số bệnh nhân duy nhất:', len(clean)-clean['patient_nbr'].nunique())

## 6. Tích hợp dữ liệu tham chiếu và chuẩn hóa danh mục

Các cột mã `admission_type_id`, `discharge_disposition_id`, `admission_source_id` là **mã danh mục**, không phải đại lượng liên tục. `IDS_mapping.csv` là file tham chiếu để giải nghĩa mã; không thực hiện nối trực tiếp khi chưa chuẩn hóa cấu trúc từng bảng tra cứu trong file đó. Ở bước kho dữ liệu sẽ dựng các dimension/mapping rõ ràng.

Ở đây chỉ tạo bản sao dạng chuỗi của các mã để nhóm tránh hiểu nhầm là số đo trong các phân tích phân nhóm. Không dùng mã discharge như đầu vào dự đoán nếu bài toán dự đoán ở thời điểm trước khi thông tin đó khả dụng.

In [ ]:
id_category_cols = ['admission_type_id','discharge_disposition_id','admission_source_id']
for col in id_category_cols:
    clean[col+'_category'] = clean[col].astype('string')
ref_file=root/'data'/'raw'/'IDS_mapping.csv'
print('File mã tham chiếu:', 'Đã có' if ref_file.exists() else 'Chưa tìm thấy', ref_file)
print('Các cột danh mục được tạo:', [c+'_category' for c in id_category_cols])
print('Không gộp trực tiếp IDS_mapping.csv vì chứa nhiều bảng mã khác nhau.')

## 7. Kiểm tra sau xử lý, lưu cleaned CSV và báo cáo so sánh

In [ ]:
clean['readmitted_30d'] = clean['readmitted'].eq('<30').astype('int8')
assert len(clean)==len(df), 'Số lượt khám thay đổi bất ngờ'
assert clean['encounter_id'].equals(df['encounter_id']), 'ID lượt khám đã thay đổi'
assert clean['patient_nbr'].equals(df['patient_nbr']), 'ID bệnh nhân đã thay đổi'
assert clean['readmitted_30d'].equals(computed_target), 'Nhãn mục tiêu bị sai'
assert 'weight' not in clean.columns
assert not clean.select_dtypes(include=['object','string']).apply(
    lambda c: c.astype('string').str.strip().eq('?').fillna(False).any()).any(), 'Còn dấu ?'

comparison = pd.DataFrame([
 {'chi_so':'Số lượt khám','truoc':len(df),'sau':len(clean)},
 {'chi_so':'Số bệnh nhân duy nhất','truoc':df.patient_nbr.nunique(),'sau':clean.patient_nbr.nunique()},
 {'chi_so':'Số cột','truoc':df.shape[1],'sau':clean.shape[1]},
 {'chi_so':'Số encounter_id trùng','truoc':df.encounter_id.duplicated().sum(),'sau':clean.encounter_id.duplicated().sum()},
 {'chi_so':'Số trường hợp dưới 30 ngày','truoc':int(computed_target.sum()),'sau':int(clean.readmitted_30d.sum())}
])
display(comparison)
comparison.to_csv(out_report/'before_after_summary.csv',index=False,encoding='utf-8-sig')
clean_path=out_data/'step3_cleaned.csv'
clean.to_csv(clean_path,index=False,encoding='utf-8-sig')
print('Đã lưu:',clean_path)
print('Kích thước:',clean.shape)
print('Kiểm tra số dòng, ID, nhãn, dấu ?: ĐẠT')

## 8. Nhật ký quyết định — bắt buộc cho rubric

In [ ]:
decision_log=pd.DataFrame([
 ['D03-01','Chuyển dấu ? và chuỗi trống thành missing','Dữ liệu gốc mã hóa thiếu bằng ký hiệu, pandas không nhận ra', 'Số ô thiếu theo cột, tỷ lệ thiếu', 'Giữ raw, xuất bảng missing_symbols'],
 ['D03-02','Loại weight khỏi bản cleaned (không xóa khỏi raw)', f'Thiếu {before_weight_na}/{len(df)} (~{100*before_weight_na/len(df):.2f}%)', 'Giảm cột đầu vào, không mất lượt khám', 'Thử giữ nhóm available ở phân tích độ nhạy nếu cần'],
 ['D03-03','Gán Unknown cho specialty, payer, race, diagnosis bị thiếu', 'Tránh suy diễn danh mục thiếu', 'Thay số NaN ở các cột danh mục', 'Cân nhắc thay đổi chiến lược theo validation'],
 ['D03-04','Giữ None cho A1C và glucose, thêm cờ tested','Không làm xét nghiệm không đồng nghĩa kết quả bình thường','Phân bố nhóm None; 2 cờ xét nghiệm', 'Không sử dụng cờ nếu xét nghiệm chưa khả dụng tại thời điểm dự đoán'],
 ['D03-05','Giữ các lượt khám của cùng bệnh nhân', 'Các lượt khám là sự kiện riêng', 'Số dòng không đổi', 'Bắt buộc GroupShuffleSplit hoặc StratifiedGroupKFold theo patient_nbr khi train/test'],
 ['D03-06','Không xóa outlier hàng loạt', 'Giá trị cực đoan có thể có ý nghĩa', 'Giữ nguyên các bản ghi', 'Xác minh bất hợp lệ bằng range_checks'],
 ['D03-07','Sao chép mã ID dưới dạng danh mục, chưa nối IDS_mapping','File tham chiếu có nhiều loại bảng mã', 'Thêm 3 cột *_category', 'Dựng mapping đúng loại trong bước star schema'],
 ['D03-08','Chưa fit scaler, encoder, imputer', 'Fit trước tách tập gây rò rỉ dữ liệu', 'Chưa có ma trận đặc trưng huấn luyện', 'Dùng sklearn Pipeline sau khi chia theo patient_nbr'],
 ],columns=['ma_quyet_dinh','quyet_dinh','ly_do','anh_huong','kiem_tra_hoac_hanh_dong_tiep'])
display(decision_log)
decision_log.to_csv(out_report/'decision_log_step3.csv',index=False,encoding='utf-8-sig')
print('Đã lưu:',out_report/'decision_log_step3.csv')

## 9. Tổng kết & bàn giao

**Đầu ra:**
- `data/processed/step3_cleaned.csv` — dữ liệu đã chuẩn hóa (chưa tạo ma trận huấn luyện)
- `reports/preprocessing/missing_symbols_before_cleaning.csv`
- `reports/preprocessing/range_checks.csv`
- `reports/preprocessing/before_after_summary.csv`
- `reports/preprocessing/decision_log_step3.csv`

**Chuẩn bị Bước 4:** biến đổi & giảm chiều; chọn đặc trưng theo thời điểm dự đoán; tránh rò rỉ nhãn; sau đó tách train/test **theo bệnh nhân** trước khi học scaler/encoder/imputer. Giữ `patient_nbr` để chia nhóm nhưng không dùng nó làm đặc trưng đầu vào.